In [2]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain
from langchain.schema import BaseOutputParser
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

False

In [3]:
import os
from langchain.chat_models import ChatOpenAI

os.environ["OPENAI_API_KEY"] = ""  # Replace with your OpenRouter API key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    model="qwen/qwen-2.5-72b-instruct",
    temperature=0.3,
    max_tokens=4000,
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    openai_api_base=os.environ.get("OPENAI_API_BASE")
)

print("✅ OpenRouter LLM configured successfully!")
print(f"🤖 Using model: {llm.model_name}")

✅ OpenRouter LLM configured successfully!
🤖 Using model: qwen/qwen-2.5-72b-instruct


In [4]:
def load_reviewer_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load reviewer data from multiple repositories
reviewer_data_paths = {
    'jquery': r'PR data for Reviewers/jquery_jquery_reviewer_data_train.json'
}

all_reviewer_data = {}
total_prs = 0

print("📁 Loading reviewer data from multiple repositories...")
print("=" * 50)

for repo_name, file_path in reviewer_data_paths.items():
    if os.path.exists(file_path):
        print(f"📂 Loading {repo_name} reviewer data from: {file_path}")
        data = load_reviewer_data(file_path)
        
        if data and 'pr_reviewer_data' in data:
            all_reviewer_data[repo_name] = data
            pr_count = len(data['pr_reviewer_data'])
            total_prs += pr_count
            print(f"   ✅ Loaded {pr_count} PRs from {repo_name}")
        else:
            print(f"   ❌ Failed to load {repo_name} data")
            all_reviewer_data[repo_name] = {'pr_reviewer_data': []}
    else:
        print(f"   ⚠️  File not found: {file_path}")
        all_reviewer_data[repo_name] = {'pr_reviewer_data': []}

print("=" * 50)
print(f"📊 Total loaded: {total_prs} PRs from {len([k for k in all_reviewer_data if all_reviewer_data[k]['pr_reviewer_data']])} repositories")

combined_reviewer_data = {'pr_reviewer_data': []}
for repo_name, data in all_reviewer_data.items():
    for pr in data.get('pr_reviewer_data', []):
        pr['repo_name'] = repo_name
        combined_reviewer_data['pr_reviewer_data'].append(pr)

reviewer_data = combined_reviewer_data
print(f"🔗 Combined dataset: {len(reviewer_data['pr_reviewer_data'])} total PRs for analysis")

📁 Loading reviewer data from multiple repositories...
📂 Loading jquery reviewer data from: PR data for Reviewers/jquery_jquery_reviewer_data_train.json
   ✅ Loaded 268 PRs from jquery
📊 Total loaded: 268 PRs from 1 repositories
🔗 Combined dataset: 268 total PRs for analysis


In [5]:
def analyze_reviewer_activity(reviewer_data):
    if not reviewer_data or 'pr_reviewer_data' not in reviewer_data:
        return {}
    
    reviewer_stats = {}
    
    for pr in reviewer_data['pr_reviewer_data']:
        for review in pr.get('reviews', []):
            username = review.get('reviewer_username')
            if username not in reviewer_stats:
                reviewer_stats[username] = {
                    'total_reviews': 0,
                    'total_comments': 0,
                    'total_prs': 0,
                    'repos': set(),
                    'review_details': []
                }
            
            reviewer_stats[username]['total_reviews'] += 1
            reviewer_stats[username]['repos'].add(pr.get('repo_name', ''))
            reviewer_stats[username]['review_details'].append({
                'pr_number': pr.get('pr_number'),
                'title': pr.get('title', ''),
                'description': pr.get('description', ''),
                'review_state': review.get('state'),
                'review_body': review.get('body', ''),
                'labels': pr.get('labels', []),
                'additions': pr.get('additions', 0),
                'deletions': pr.get('deletions', 0),
                'changed_files_count': pr.get('changed_files_count', 0)
            })
        
        for comment in pr.get('review_comments', []):
            username = comment.get('reviewer_username')
            if username in reviewer_stats:
                reviewer_stats[username]['total_comments'] += 1
    
    for username in reviewer_stats:
        reviewer_stats[username]['repos'] = list(reviewer_stats[username]['repos'])
        reviewer_stats[username]['total_prs'] = len(set(review['pr_number'] for review in reviewer_stats[username]['review_details']))
    
    return reviewer_stats

if reviewer_data:
    reviewer_stats = analyze_reviewer_activity(reviewer_data)
    print(f"📊 Analyzed {len(reviewer_stats)} reviewers")
else:
    reviewer_stats = {}

📊 Analyzed 29 reviewers


In [6]:
class ReviewerProfile(BaseModel):
    reviewer_name: str = Field(description="Reviewer's username")
    experience_level: str = Field(description="Estimated experience level (Junior/Mid/Senior/Lead)")
    primary_skills: List[str] = Field(description="List of primary JavaScript technical skills")
    programming_languages: List[str] = Field(description="Programming languages expertise (JavaScript-focused)")
    summary: str = Field(description="Overall JavaScript reviewer profile summary")
    javascript_skill_matrix: Dict[str, List[str]] = Field(description="Detailed JavaScript skill assessment with frequency embedded in both category keys and individual skill values (e.g., 'Framework/Library Expertise, frequency: 10': ['Node.js, frequency: 1', 'Express.js, frequency: 5'])")

profile_parser = PydanticOutputParser(pydantic_object=ReviewerProfile)

js_skill_categories = {
    "Framework/Library Expertise": [
        "Frontend Frameworks: React, Angular, Vue.js, Svelte, Ember.js, Backbone.js, Mithril, Preact, jQuery",
        "Backend Frameworks: Node.js, Express.js, Koa.js",
        "State Management: Redux, Context API, Zustand",
        "Testing Libraries: Jest, Mocha, Chai",
        "Static Site Generation/Server-Side Rendering: Next.js, Nuxt.js"
    ],
    "Asynchronous Programming": ["async/await", "Promises", "Callbacks", "Event Loop", "Concurrency Control"],
    "API Design & Consumption": ["RESTful APIs", "GraphQL", "WebSockets", "Authentication & Authorization", "API Error Handling", "Rate Limiting"],
    "Error Handling": ["Try-Catch", "Custom Error Classes", "Error Boundaries"],
    "Frontend Development Skills": ["DOM Manipulation"],
    "Backend Development Skills": ["Server-Side Logic", "Database Integration", "API Endpoints", "Authentication/Authorization", "Middleware", "Microservices", "Caching"],
    "DevOps & Deployment": ["CI/CD Pipelines", "Containerization", "Cloud Providers", "Load Balancing", "Web Servers"],
    "Advanced JavaScript Concepts": ["Closures", "Higher-Order Functions", "Prototypes & Inheritance", "Module Systems", "Event Delegation", "Memory Management"],
    "Functional Programming": ["Immutability", "Pure Functions", "Declarative Programming", "Composition"],
    "Data Structures & Algorithms": ["Data Structures", "Searching and Sorting Algorithms", "Recursion", "Big O Notation"],
    "TypeScript": ["Type Definitions", "Generics", "Type Inference", "Modules & Namespaces", "Type Narrowing"],
    "Progressive Web Apps (PWA)": ["Service Workers", "Web Push Notifications", "Caching Strategies", "App Shell Model", "Manifest File"],
    "Mobile Development with JavaScript": ["React Native", "Ionic", "Cordova/PhoneGap"],
    "Web Performance Optimization": ["Lazy Loading", "Code Splitting", "Minification & Compression", "Critical Rendering Path Optimization", "Preloading & Prefetching"],
    "Event-Driven Architecture": ["Event Emitters", "Pub/Sub Model"],
    "Dependency Management": ["npm/yarn", "Package-lock.json & Yarn.lock"],
    "Graphical Data Visualization": ["D3.js", "Chart.js", "WebGL & Three.js"]
}

# List of valid category names for strict validation
VALID_CATEGORY_NAMES = list(js_skill_categories.keys())

skills_analysis_prompt = PromptTemplate(
    input_variables=["review_data", "reviewer_name"],
    template="""Analyze the following GitHub Review data for reviewer {reviewer_name} and extract their JavaScript technical skills and expertise:

Review Data:
{review_data}

Focus specifically on JavaScript-related skills. Based on this review data, identify:
1. JavaScript frameworks and libraries they show expertise in reviewing (React, Vue, Angular, Node.js, etc.)
2. JavaScript programming patterns and paradigms they understand
3. Frontend vs Backend JavaScript knowledge demonstrated in reviews
4. Testing frameworks and methodologies knowledge in JavaScript
5. Build tools and development workflow understanding
6. JavaScript ES6+ features knowledge
7. TypeScript usage and proficiency shown in reviews
8. API development and consumption patterns knowledge
9. Asynchronous programming understanding
10. Code quality and modern JavaScript practices knowledge

Provide a detailed analysis focusing exclusively on JavaScript ecosystem skills and expertise.
Look for evidence in review comments, review decisions, and technical feedback patterns.""",
    partial_variables={"format_instructions": "Provide a detailed JavaScript-focused technical analysis."}
)

profile_generation_prompt = PromptTemplate(
    input_variables=["skills_analysis", "reviewer_name", "review_count", "repo_list"],
    template="""Based on the following JavaScript skills analysis, generate a comprehensive JavaScript reviewer profile:

Reviewer: {reviewer_name}
Total Reviews: {review_count}
Repositories: {repo_list}

Skills Analysis:
{skills_analysis}

VALID CATEGORY NAMES FOR skill_frequency (THESE ARE THE ONLY ALLOWED KEYS):
{valid_categories}

JavaScript Skill Matrix Details:
{js_categories}

Generate a JavaScript reviewer profile that includes:
1. Experience level assessment (Junior/Mid/Senior/Lead) based on JavaScript review quality
2. Primary JavaScript technical skills and frameworks they show expertise in
3. Overall JavaScript reviewer summary
4. Detailed skill matrix mapping to the provided JavaScript categories WITH EMBEDDED FREQUENCY

========== CRITICAL RULES FOR javascript_skill_matrix ==========
YOU MUST FOLLOW THESE RULES EXACTLY OR THE OUTPUT WILL BE REJECTED:

FREQUENCY EMBEDDING RULES:
1. For each category that has skills, ADD the frequency count to the category name in the KEY
2. Format: "Category Name, frequency: X" where X is the total count for that category
3. For each individual skill, ALSO ADD the frequency count to the skill name in the VALUE
4. Format: "Skill Name, frequency: Y" where Y is the count for that specific skill
5. For categories with NO skills (empty arrays), DO NOT add frequency - just use the category name alone

CATEGORY NAME RULES:
1. ONLY use these EXACT base category names:
   - "Framework/Library Expertise"
   - "Asynchronous Programming"
   - "API Design & Consumption"
   - "Error Handling"
   - "Frontend Development Skills"
   - "Backend Development Skills"
   - "DevOps & Deployment"
   - "Advanced JavaScript Concepts"
   - "Functional Programming"
   - "Data Structures & Algorithms"
   - "TypeScript"
   - "Progressive Web Apps (PWA)"
   - "Mobile Development with JavaScript"
   - "Web Performance Optimization"
   - "Event-Driven Architecture"
   - "Dependency Management"
   - "Graphical Data Visualization"

2. DO NOT and NEVER EVER create any new category names
3. DO NOT and NEVER EVER use subcategory names like "Testing Libraries" or "Backend Frameworks"
4. DO NOT and NEVER EVER use individual skill names like "Node.js", "Express.js", "CI/CD" as category names
5. If you see skills related to testing (Jest, Mocha, etc.), count them under "Framework/Library Expertise"
6. If you see skills related to code quality, linting, etc., count them appropriately under existing categories

Example CORRECT javascript_skill_matrix:
{{
  "Framework/Library Expertise, frequency: 15": ["Node.js, frequency: 6", "Express.js, frequency: 5", "Jest, frequency: 4"],
  "Asynchronous Programming, frequency: 5": ["Promises, frequency: 3", "async/await, frequency: 2"],
  "API Design & Consumption, frequency: 10": ["RESTful APIs, frequency: 10"],
  "Backend Development Skills, frequency: 20": ["Middleware, frequency: 8", "API Endpoints, frequency: 7", "Authentication/Authorization, frequency: 5"],
  "DevOps & Deployment, frequency: 10": ["CI/CD Pipelines, frequency: 6", "Containerization, frequency: 4"],
  "Advanced JavaScript Concepts, frequency: 5": ["Prototypes & Inheritance, frequency: 3", "Module Systems, frequency: 2"],
  "Dependency Management, frequency: 10": ["npm/yarn, frequency: 10"],
  "Error Handling": [],
  "Frontend Development Skills": [],
  "Functional Programming": [],
  "Data Structures & Algorithms": [],
  "TypeScript": [],
  "Progressive Web Apps (PWA)": [],
  "Mobile Development with JavaScript": [],
  "Web Performance Optimization": [],
  "Event-Driven Architecture": [],
  "Graphical Data Visualization": []
}}

Example WRONG javascript_skill_matrix (DO NOT DO THIS):
{{
  "Testing Libraries, frequency: 5": ["Jest, frequency: 5"],  ← WRONG! "Testing Libraries" is not a main category
  "Node.js": ["Express.js"],  ← WRONG! "Node.js" is a skill, not a category
  "Framework/Library Expertise": ["Node.js", "Express.js"],  ← WRONG! Missing frequency for non-empty array AND individual skills
  "Framework/Library Expertise, frequency: 10": ["Node.js", "Express.js"]  ← WRONG! Missing frequency for individual skills
}}
========== END OF CRITICAL RULES FOR FREQUENCY EMBEDDING ==========

========== CRITICAL RULES FOR SKILL SELECTION ==========
YOU MUST FOLLOW THESE RULES EXACTLY FOR THE SKILL MATRIX:

1. The javascript_skill_matrix MUST use the EXACT 17 category names as keys (same as skill_frequency)
2. For each category, ONLY include skills that are EXPLICITLY LISTED in the subcategory definitions above
3. DO NOT add skills that are not in the predefined lists (e.g., "path-to-regexp", "HTTP/2", etc.)
4. If no evidence found for a category, use an empty array []
5. DO NOT invent or add new skills - ONLY use skills from the predefined lists

Example CORRECT javascript_skill_matrix:
{{
  "Framework/Library Expertise, frequency: 15": ["Node.js, frequency: 6", "Express.js, frequency: 5", "Jest, frequency: 4"],
  "API Design & Consumption, frequency: 10": ["RESTful APIs, frequency: 8", "Authentication & Authorization, frequency: 2"],
  "Backend Development Skills, frequency: 20": ["Middleware, frequency: 8", "API Endpoints, frequency: 12"],
  "Dependency Management, frequency: 10": ["npm/yarn, frequency: 10"],
  "TypeScript": []
}}

Example WRONG javascript_skill_matrix (DO NOT DO THIS):
{{
  "Framework/Library Expertise": ["Node.js", "Express.js", "path-to-regexp"],  ← WRONG! "path-to-regexp" not in list AND missing frequencies
  "API Design & Consumption, frequency: 10": ["RESTful APIs", "HTTP/2"],  ← WRONG! "HTTP/2" not in list AND missing individual frequencies
  "DevOps & Deployment": ["CI/CD Pipelines", "Dependency Management"]  ← WRONG! "Dependency Management" is a category, not a skill, AND missing frequencies
}}
========== END OF CRITICAL RULES ==========

{format_instructions}

Be specific and evidence-based in your JavaScript skill assessment.
Map identified skills to the appropriate categories in the skill matrix.
Count category usage frequency based on evidence from PR titles, descriptions, and review patterns.
REMEMBER: 
- Embed frequency counts in BOTH the category names AND individual skill names for non-empty skill arrays!
- Only use the 17 exact base category names!
- Only use skills EXPLICITLY listed in the subcategory definitions for javascript_skill_matrix!
- DO NOT invent new skills or add skills not in the predefined lists!
- Format: "Category Name, frequency: X": ["skill1, frequency: Y", "skill2, frequency: Z"]
- The category frequency should be the SUM of all individual skill frequencies in that category
""",
    partial_variables={
        "format_instructions": profile_parser.get_format_instructions(),
        "js_categories": "\n".join([f"{cat}: {', '.join(skills)}" for cat, skills in js_skill_categories.items()]),
        "valid_categories": "\n".join([f"   - \"{cat}\"" for cat in VALID_CATEGORY_NAMES])
    }
)


In [7]:
def create_analysis_chains(llm):
    
    skills_chain = LLMChain(
        llm=llm,
        prompt=skills_analysis_prompt,
        verbose=False  # Changed from True to False
    )
    
    profile_chain = LLMChain(
        llm=llm,
        prompt=profile_generation_prompt,
        verbose=False  # Changed from True to False
    )
    
    return skills_chain, profile_chain

def prepare_review_data_for_analysis(reviewer_data, max_reviews=20):
    """Prepare review data for LLM analysis (limit size to avoid token limits)"""
    
    review_details = reviewer_data['review_details'][:max_reviews] 
    
    review_summary = []
    for review in review_details:
        summary = {
            'pr_title': review['title'][:100],
            'repo': review.get('repo', ''),
            'labels': review['labels'],
            'review_state': review['review_state'],
            'additions': review['additions'],
            'deletions': review['deletions'],
            'files_changed': review['changed_files_count']
        }
        
        if review['description'] and len(review['description']) < 500:
            summary['pr_description'] = review['description'][:300]
        
        if review['review_body'] and len(review['review_body']) < 500:
            summary['review_comment'] = review['review_body'][:300]
        
        review_summary.append(summary)
    
    return review_summary

if llm and reviewer_stats:
    print(f"🔧 Creating analysis chains for ALL reviewers...")
    
    skills_chain, profile_chain = create_analysis_chains(llm)
    
    eligible_reviewers = [
        (name, stats) for name, stats in reviewer_stats.items()
        if stats['total_reviews'] >= 2 and not name.endswith('[bot]') and 'bot' not in name.lower()
    ]
    
    eligible_reviewers.sort(key=lambda x: x[1]['total_reviews'], reverse=True)
    
    print(f"✅ Chains created successfully")
    print(f"👥 Found {len(eligible_reviewers)} eligible reviewers")
    print(f"📊 Will process reviewers with 2+ reviews (excluding bots)")
    
elif llm:
    print("⚠️  LLM configured but no reviewers found")
    print("Please check that reviewer data was loaded correctly")
else:
    print("⚠️  LLM not configured - please run the LLM configuration cell first")
    print("Set up your OpenRouter API key to continue")

🔧 Creating analysis chains for ALL reviewers...
✅ Chains created successfully
👥 Found 15 eligible reviewers
📊 Will process reviewers with 2+ reviews (excluding bots)


In [8]:
def generate_reviewer_profile(reviewer_name, reviewer_data, review_summary):
    try:
        skills_analysis_input = {
            "reviewer_name": reviewer_name,
            "review_data": json.dumps(review_summary, indent=2)
        }
        
        print(f"  🔍 Analyzing skills for {reviewer_name}...")
        skills_analysis_result = skills_chain.run(skills_analysis_input)
        
        profile_input = {
            "reviewer_name": reviewer_name,
            "skills_analysis": skills_analysis_result,
            "review_count": reviewer_data['total_reviews'],
            "repo_list": ', '.join(reviewer_data['repos'])
        }
        
        print(f"  🤖 Generating profile for {reviewer_name}...")
        profile_result = profile_chain.run(profile_input)
        
        try:
            parsed_profile = profile_parser.parse(profile_result)
            profile_dict = parsed_profile.dict()
        except Exception as parse_error:
            print(f"  ⚠️  Profile parsing failed for {reviewer_name}: {parse_error}")
            profile_dict = {
                "reviewer_name": reviewer_name,
                "raw_response": profile_result,
                "parsing_error": str(parse_error)
            }
        
        return {
            "success": True,
            "profile": profile_dict
            # Removed skills_analysis and raw_profile from return to reduce output
        }
        
    except Exception as e:
        print(f"  ❌ Error generating profile for {reviewer_name}: {e}")
        return {
            "success": False,
            "error": str(e),
            "reviewer": reviewer_name
        }

In [9]:
import time

def load_checkpoint(checkpoint_file="reviewer_profile_generation_checkpoint.json"):
    """Load the last checkpoint to resume processing"""
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            return json.load(f)
    return {"processed_reviewers": [], "profiles": {}}

def save_checkpoint(data, checkpoint_file="reviewer_profile_generation_checkpoint.json"):
    """Save checkpoint to resume processing later"""
    with open(checkpoint_file, 'w') as f:
        json.dump(data, f, indent=2)

def save_profile_to_file(reviewer_name, profile_data, timestamp):
    """Save individual profile to file"""
    output_dir = "Reviewers Profiles"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    filename = f"{reviewer_name}_reviewer_profile_{timestamp}.json"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(profile_data, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved profile to {filepath}")
    return filepath

def run_profile_generation_session(batch_size=15, delay_seconds=2):
    """Run profile generation in batches with checkpoint support"""
    global eligible_reviewers, skills_chain, profile_chain
    
    if not eligible_reviewers:
        print("❌ No eligible reviewers found for processing")
        return []
    
    # Load checkpoint
    checkpoint_data = load_checkpoint()
    processed_reviewers = set(checkpoint_data.get("processed_reviewers", []))
    all_profiles = checkpoint_data.get("profiles", {})
    
    # Find reviewers to process
    reviewers_to_process = [
        (name, data) for name, data in eligible_reviewers 
        if name not in processed_reviewers
    ]
    
    if not reviewers_to_process:
        print("✅ All eligible reviewers have already been processed!")
        return []
    
    # Take only batch_size reviewers
    batch_reviewers = reviewers_to_process[:batch_size]
    
    print(f"🚀 Starting batch processing of {len(batch_reviewers)} reviewers...")
    print(f"📊 {len(processed_reviewers)} already processed, {len(reviewers_to_process)} remaining")
    print(f"⏱️  Delay between profiles: {delay_seconds} seconds\n")
    
    batch_results = []
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for i, (reviewer_name, reviewer_data) in enumerate(batch_reviewers, 1):
        print(f"\n🔄 Processing reviewer {i}/{len(batch_reviewers)}: {reviewer_name}")
        print(f"📈 Stats: {reviewer_data['total_reviews']} reviews, {reviewer_data['total_prs']} PRs")
        
        try:
            review_summary = prepare_review_data_for_analysis(reviewer_data)
            
            if not review_summary:
                print(f"  ⚠️  No review data available for {reviewer_name}")
                continue
            
            result = generate_reviewer_profile(reviewer_name, reviewer_data, review_summary)
            
            if result["success"]:
                # Create stats without review_details
                stats_without_details = {
                    'total_reviews': reviewer_data['total_reviews'],
                    'total_comments': reviewer_data['total_comments'],
                    'total_prs': reviewer_data['total_prs'],
                    'repos': reviewer_data['repos']
                }
                
                # Add stats to profile (simplified version without verbose analysis)
                profile_data = {
                    "reviewer_name": reviewer_name,
                    "generated_at": timestamp,
                    "stats": stats_without_details,
                    "profile": result["profile"]
                }
                
                # Save individual profile
                save_profile_to_file(reviewer_name, profile_data, timestamp)
                
                # Add to collection
                all_profiles[reviewer_name] = profile_data
                processed_reviewers.add(reviewer_name)
                
                # Add to batch results
                batch_results.append({
                    "reviewer": reviewer_name,
                    "result": result,
                    "profile_data": profile_data
                })
                
                print(f"  ✅ Profile generated successfully for {reviewer_name}")
                
                # Show brief summary only
                if "profile" in result and "experience_level" in result["profile"]:
                    experience = result["profile"]["experience_level"]
                    skills_count = len(result["profile"].get("primary_skills", []))
                    print(f"  📝 {experience} level, {skills_count} primary skills identified")
                
            else:
                print(f"  ❌ Failed to generate profile for {reviewer_name}: {result.get('error', 'Unknown error')}")
            
            # Save checkpoint after each successful profile
            checkpoint_data = {
                "processed_reviewers": list(processed_reviewers),
                "profiles": all_profiles
            }
            save_checkpoint(checkpoint_data)
            
            # Rate limiting
            if i < len(batch_reviewers):  # Don't delay after the last one
                time.sleep(delay_seconds)
            
        except Exception as e:
            print(f"  💥 Unexpected error processing {reviewer_name}: {e}")
            continue
    
    print(f"\n🎉 Batch processing completed!")
    print(f"✅ Successfully processed {len(batch_results)} reviewers in this batch")
    print(f"📁 Individual profiles saved to 'Reviewers Profiles' directory")
    
    remaining_count = len(reviewers_to_process) - len(batch_reviewers)
    if remaining_count > 0:
        print(f"📋 {remaining_count} reviewers remaining for future batches")
    else:
        print(f"🎯 All eligible reviewers have been processed!")
    
    return batch_results

# Setup for batch processing
if 'eligible_reviewers' in locals() and eligible_reviewers:
    print(f"📊 Found {len(eligible_reviewers)} eligible reviewers for processing")
    print("🔧 Batch processing functions ready!")
    print("Run the batch processing cell below to start generating profiles")
else:
    print("❌ No eligible reviewers found for processing")
    print("Please ensure reviewer data is loaded and LLM is configured")

📊 Found 15 eligible reviewers for processing
🔧 Batch processing functions ready!
Run the batch processing cell below to start generating profiles


In [10]:
BATCH_SIZE = 20       # Number of reviewers to process in each batch (reduced for initial test)
DELAY_SECONDS = 2      # Delay between each profile generation


batch_results = run_profile_generation_session(
    batch_size=BATCH_SIZE, 
    delay_seconds=DELAY_SECONDS
)

if batch_results:
    print(f"\n📋 Quick Preview of This Batch:")
    for i, profile_data in enumerate(batch_results[:3], 1):
        reviewer_name = profile_data["reviewer"]
        profile = profile_data["result"]["profile"]
        if profile and isinstance(profile, dict):
            experience = profile.get('experience_level', 'Unknown')
            skills_count = len(profile.get('primary_skills', []))
            print(f"  {i}. {reviewer_name}: {experience} level, {skills_count} skills")
        else:
            print(f"  {i}. {reviewer_name}: Profile generated (parsing may have failed)")
    
    if len(batch_results) > 3:
        print(f"  ... and {len(batch_results) - 3} more profiles generated")
else:
    print("📝 No new profiles generated in this batch")

🚀 Starting batch processing of 15 reviewers...
📊 0 already processed, 15 remaining
⏱️  Delay between profiles: 2 seconds


🔄 Processing reviewer 1/15: mgol
📈 Stats: 311 reviews, 134 PRs
  🔍 Analyzing skills for mgol...
  🤖 Generating profile for mgol...
💾 Saved profile to Reviewers Profiles\mgol_reviewer_profile_20251114_163138.json
  ✅ Profile generated successfully for mgol
  📝 Senior level, 9 primary skills identified

🔄 Processing reviewer 2/15: timmywil
📈 Stats: 292 reviews, 174 PRs
  🔍 Analyzing skills for timmywil...
  🤖 Generating profile for timmywil...
💾 Saved profile to Reviewers Profiles\timmywil_reviewer_profile_20251114_163138.json
  ✅ Profile generated successfully for timmywil
  📝 Senior level, 5 primary skills identified

🔄 Processing reviewer 3/15: gibson042
📈 Stats: 58 reviews, 39 PRs
  🔍 Analyzing skills for gibson042...
  🤖 Generating profile for gibson042...
💾 Saved profile to Reviewers Profiles\gibson042_reviewer_profile_20251114_163138.json
  ✅ Profile generated